# CleanAir AI — Notebook 03: Pollution Source Attribution and Land-use Reasoning

This notebook estimates the likely source of pollution for each monitoring station.

## Input

- `data/processed/station_weather_geospatial.csv`

## Main tasks

1. Load AQI + weather + geospatial station data.
2. Calculate pollutant ratios and normalized pollutant indicators.
3. Score likely pollution sources:
   - Vehicular traffic
   - Industrial combustion
   - Construction / road dust
   - Biomass / fine-particle combustion
   - Photochemical smog
4. Select the most likely source.
5. Calculate source attribution confidence.
6. Compare source attribution with land-use proxy.
7. Generate action recommendations.
8. Save station-level and city-level outputs.

## Important note

This notebook performs explainable rule-based attribution, not ground-truth chemical source apportionment.  
The output should be described as **source-likelihood estimation** based on available pollutant and context signals.

In [2]:
from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"

for folder in [
    PROCESSED_DIR,
    OUTPUTS_DIR,
    REPORTS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed folder:", PROCESSED_DIR)
print("Outputs folder:", OUTPUTS_DIR)
print("Reports folder:", REPORTS_DIR)

Project root: C:\Users\Lenovo\Desktop\CleanAir_AI
Processed folder: C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed
Outputs folder: C:\Users\Lenovo\Desktop\CleanAir_AI\outputs
Reports folder: C:\Users\Lenovo\Desktop\CleanAir_AI\reports


In [4]:
INPUT_PATH = PROCESSED_DIR / "station_weather_geospatial.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing input file: {INPUT_PATH}. Run Notebook 02 first."
    )

station_df = pd.read_csv(INPUT_PATH)

print("Loaded station weather-geospatial data.")
print("Shape:", station_df.shape)

display(station_df.head())

Loaded station weather-geospatial data.
Shape: (500, 47)


,station_id,station_name,city,state,lat,lon,timestamp,snapshot_fetch_time,PM2.5,PM10,...,landuse_data_source,urban_pressure_score,urban_pressure_category,road_density_proxy,road_density_category,road_density_data_source,geospatial_data_level,environmental_risk_score,environmental_risk_category,notebook_02_processed_time
0,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-16 05:00:00,2026-07-16 05:38:19,54.0,172.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,51.86,High,2026-07-16 10:51:20
1,andhra_pradesh__anantapur__gulzarpet_anantapur...,"Gulzarpet, Anantapur - APPCB",Anantapur,Andhra Pradesh,14.675886,77.593027,2026-07-16 05:00:00,2026-07-16 05:38:19,85.0,76.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,34.66,Moderate,2026-07-16 10:51:20
2,andhra_pradesh__chittoor__gangineni_cheruvu_ch...,"Gangineni Cheruvu, Chittoor - APPCB",Chittoor,Andhra Pradesh,13.204880,79.097889,2026-07-16 05:00:00,2026-07-16 05:38:19,70.0,64.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,37.09,Moderate,2026-07-16 10:51:20
3,andhra_pradesh__eluru__district_court_eluru_appcb,"District Court, Eluru - APPCB",Eluru,Andhra Pradesh,16.711754,81.092095,2026-07-16 05:00:00,2026-07-16 05:38:19,49.0,49.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,37.99,Moderate,2026-07-16 10:51:20
4,andhra_pradesh__guntur__rajendra_nagar_north_g...,"Rajendra Nagar North, Guntur - APPCB",Guntur,Andhra Pradesh,16.316553,80.413302,2026-07-16 05:00:00,2026-07-16 05:38:19,52.0,52.0,...,Proxy from station/city naming,15,Low urban pressure,20,Low road density proxy,Proxy from city/station naming,Proxy geospatial features,31.32,Moderate,2026-07-16 10:51:20


In [5]:
required_columns = [
    "station_id",
    "station_name",
    "city",
    "state",
    "lat",
    "lon",
    "reported_aqi",
    "reported_aqi_category",
    "reported_dominant_pollutant",
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "O3",
    "NH3",
    "temperature_2m",
    "relative_humidity_2m",
    "rain",
    "wind_speed_10m",
    "cloud_cover",
    "weather_trapping_score",
    "landuse_type_proxy",
    "urban_pressure_score",
    "road_density_proxy",
    "environmental_risk_score"
]

missing_columns = [
    col for col in required_columns
    if col not in station_df.columns
]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

print("All required columns found.")

All required columns found.


In [6]:
numeric_columns = [
    "reported_aqi",
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "O3",
    "NH3",
    "temperature_2m",
    "relative_humidity_2m",
    "rain",
    "wind_speed_10m",
    "cloud_cover",
    "weather_trapping_score",
    "urban_pressure_score",
    "road_density_proxy",
    "environmental_risk_score"
]

for col in numeric_columns:
    station_df[col] = pd.to_numeric(
        station_df[col],
        errors="coerce"
    )

print("Numeric conversion complete.")
print(station_df[numeric_columns].isna().sum())

Numeric conversion complete.
reported_aqi                22
PM2.5                       37
PM10                        33
NO2                         30
SO2                         52
CO                          27
O3                          29
NH3                         72
temperature_2m               0
relative_humidity_2m         0
rain                         0
wind_speed_10m               0
cloud_cover                  0
weather_trapping_score       0
urban_pressure_score         0
road_density_proxy           0
environmental_risk_score    22
dtype: int64


In [7]:
def normalize_series(series):
    """
    Min-max normalize a series to 0-1.
    """

    series = pd.to_numeric(series, errors="coerce")

    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (series - min_val) / (max_val - min_val)


def safe_divide(numerator, denominator):
    """
    Safely divide numeric values.
    """

    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")

    return numerator / denominator.replace(0, np.nan)


print("Helper functions ready.")

Helper functions ready.


In [8]:
station_df["pm10_pm25_ratio"] = safe_divide(
    station_df["PM10"],
    station_df["PM2.5"]
)

station_df["pm25_pm10_ratio"] = safe_divide(
    station_df["PM2.5"],
    station_df["PM10"]
)

station_df["no2_co_ratio"] = safe_divide(
    station_df["NO2"],
    station_df["CO"]
)

station_df["o3_no2_ratio"] = safe_divide(
    station_df["O3"],
    station_df["NO2"]
)

ratio_columns = [
    "pm10_pm25_ratio",
    "pm25_pm10_ratio",
    "no2_co_ratio",
    "o3_no2_ratio"
]

print("Ratio features created.")
display(station_df[ratio_columns].describe())

Ratio features created.


,pm10_pm25_ratio,pm25_pm10_ratio,no2_co_ratio,o3_no2_ratio
count,451.000000,451.000000,448.000000,444.000000
mean,1.698398,0.711571,1.016082,1.947595
std,0.974757,0.304845,1.855909,3.818277
min,0.469194,0.064516,0.000000,0.000000
25%,1.140394,0.483315,0.368421,0.485555
50%,1.529412,0.653846,0.666667,0.967204
75%,2.069048,0.876894,1.091991,1.930804
max,15.500000,2.131313,24.750000,60.000000


In [9]:
signal_columns = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "O3",
    "NH3",
    "reported_aqi",
    "temperature_2m",
    "relative_humidity_2m",
    "rain",
    "wind_speed_10m",
    "cloud_cover",
    "weather_trapping_score",
    "urban_pressure_score",
    "road_density_proxy",
    "environmental_risk_score",
    "pm10_pm25_ratio",
    "pm25_pm10_ratio",
    "no2_co_ratio",
    "o3_no2_ratio"
]

for col in signal_columns:
    norm_col = f"{col}_norm"
    station_df[norm_col] = normalize_series(station_df[col])

print("Normalized signal features created.")

display(
    station_df[
        [
            "PM2.5_norm",
            "PM10_norm",
            "NO2_norm",
            "SO2_norm",
            "CO_norm",
            "O3_norm",
            "weather_trapping_score_norm",
            "road_density_proxy_norm",
            "urban_pressure_score_norm"
        ]
    ].head()
)

Normalized signal features created.


,PM2.5_norm,PM10_norm,NO2_norm,SO2_norm,CO_norm,O3_norm,weather_trapping_score_norm,road_density_proxy_norm,urban_pressure_score_norm
0,0.160606,0.500000,0.271186,0.177778,0.582524,0.045936,0.274533,0.0,0.0
1,0.254545,0.203704,0.135593,0.122222,0.203883,0.109541,0.220461,0.0,0.0
2,0.209091,0.166667,0.262712,0.233333,0.194175,0.137809,0.339453,0.0,0.0
3,0.145455,0.120370,0.186441,0.144444,0.165049,0.021201,0.441255,0.0,0.0
4,0.154545,0.129630,0.355932,0.155556,0.135922,0.088339,0.304573,0.0,0.0


In [10]:
landuse_text = (
    station_df["landuse_type_proxy"]
    .astype(str)
    .str.lower()
)

station_df["is_traffic_landuse"] = landuse_text.str.contains(
    "traffic|commercial|dense urban",
    regex=True
).astype(int)

station_df["is_industrial_landuse"] = landuse_text.str.contains(
    "industrial",
    regex=False
).astype(int)

station_df["is_general_urban_landuse"] = landuse_text.str.contains(
    "general urban|semi-urban|dense urban",
    regex=True
).astype(int)

print("Land-use indicators created.")

print(station_df[
    [
        "is_traffic_landuse",
        "is_industrial_landuse",
        "is_general_urban_landuse"
    ]
].sum())

Land-use indicators created.
is_traffic_landuse          166
is_industrial_landuse        49
is_general_urban_landuse    437
dtype: int64


In [11]:
def calculate_source_scores(row):
    """
    Calculate source likelihood scores from pollutant, weather,
    land-use, road-density, and urban-pressure signals.
    Scores are bounded between 0 and 100.
    """

    pm25 = row.get("PM2.5_norm", 0)
    pm10 = row.get("PM10_norm", 0)
    no2 = row.get("NO2_norm", 0)
    so2 = row.get("SO2_norm", 0)
    co = row.get("CO_norm", 0)
    o3 = row.get("O3_norm", 0)

    pm10_pm25_ratio = row.get("pm10_pm25_ratio_norm", 0)
    pm25_pm10_ratio = row.get("pm25_pm10_ratio_norm", 0)

    temp = row.get("temperature_2m_norm", 0)
    rain = row.get("rain_norm", 0)
    wind = row.get("wind_speed_10m_norm", 0)
    trapping = row.get("weather_trapping_score_norm", 0)
    road = row.get("road_density_proxy_norm", 0)
    urban = row.get("urban_pressure_score_norm", 0)

    is_traffic = row.get("is_traffic_landuse", 0)
    is_industrial = row.get("is_industrial_landuse", 0)
    is_general_urban = row.get("is_general_urban_landuse", 0)

    vehicular_score = (
        0.30 * no2
        + 0.20 * co
        + 0.20 * road
        + 0.15 * urban
        + 0.15 * is_traffic
    ) * 100

    industrial_score = (
        0.35 * so2
        + 0.20 * no2
        + 0.15 * co
        + 0.20 * is_industrial
        + 0.10 * urban
    ) * 100

    construction_road_dust_score = (
        0.35 * pm10
        + 0.20 * pm10_pm25_ratio
        + 0.15 * road
        + 0.10 * urban
        + 0.10 * wind
        + 0.10 * (1 - rain)
    ) * 100

    biomass_fine_particle_score = (
        0.35 * pm25
        + 0.20 * co
        + 0.15 * pm25_pm10_ratio
        + 0.15 * trapping
        + 0.10 * is_general_urban
        + 0.05 * (1 - wind)
    ) * 100

    photochemical_smog_score = (
        0.40 * o3
        + 0.20 * temp
        + 0.15 * no2
        + 0.10 * urban
        + 0.10 * (1 - rain)
        + 0.05 * (1 - row.get("cloud_cover_norm", 0))
    ) * 100

    scores = {
        "Vehicular traffic": vehicular_score,
        "Industrial combustion": industrial_score,
        "Construction / road dust": construction_road_dust_score,
        "Biomass / fine-particle combustion": biomass_fine_particle_score,
        "Photochemical smog": photochemical_smog_score
    }

    scores = {
        key: round(float(np.clip(value, 0, 100)), 2)
        for key, value in scores.items()
    }

    return scores


print("Source score function ready.")

Source score function ready.


In [12]:
source_score_rows = []

for _, row in station_df.iterrows():
    scores = calculate_source_scores(row)
    source_score_rows.append(scores)

source_scores_df = pd.DataFrame(source_score_rows)

source_scores_df.columns = [
    "source_score_" + col.lower()
    .replace(" ", "_")
    .replace("/", "")
    .replace("-", "_")
    for col in source_scores_df.columns
]

station_df = pd.concat(
    [station_df.reset_index(drop=True), source_scores_df.reset_index(drop=True)],
    axis=1
)

source_score_columns = list(source_scores_df.columns)

print("Source scores added:")
print(source_score_columns)

display(station_df[source_score_columns].head())

Source scores added:
['source_score_vehicular_traffic', 'source_score_industrial_combustion', 'source_score_construction__road_dust', 'source_score_biomass__fine_particle_combustion', 'source_score_photochemical_smog']


,source_score_vehicular_traffic,source_score_industrial_combustion,source_score_construction__road_dust,source_score_biomass__fine_particle_combustion,source_score_photochemical_smog
0,19.79,20.38,38.40,34.56,29.28
1,8.15,10.05,24.84,35.37,29.84
2,11.76,16.33,21.27,36.34,33.72
3,8.89,11.26,20.35,34.09,26.13
4,13.40,14.60,21.99,31.11,32.25


In [13]:
source_score_columns = [
    col for col in station_df.columns
    if col.startswith("source_score_")
]

print("Source score columns found:")
print(source_score_columns)

for col in source_score_columns:
    station_df[col] = pd.to_numeric(
        station_df[col],
        errors="coerce"
    )

print("\nMissing values before fix:")
print(station_df[source_score_columns].isna().sum())

station_df[source_score_columns] = station_df[source_score_columns].fillna(0)

print("\nMissing values after fix:")
print(station_df[source_score_columns].isna().sum())

print("\nRows where all source scores are zero:")
print((station_df[source_score_columns].sum(axis=1) == 0).sum())

Source score columns found:
['source_score_vehicular_traffic', 'source_score_industrial_combustion', 'source_score_construction__road_dust', 'source_score_biomass__fine_particle_combustion', 'source_score_photochemical_smog']

Missing values before fix:
source_score_vehicular_traffic                    52
source_score_industrial_combustion                83
source_score_construction__road_dust              49
source_score_biomass__fine_particle_combustion    69
source_score_photochemical_smog                   53
dtype: int64

Missing values after fix:
source_score_vehicular_traffic                    0
source_score_industrial_combustion                0
source_score_construction__road_dust              0
source_score_biomass__fine_particle_combustion    0
source_score_photochemical_smog                   0
dtype: int64

Rows where all source scores are zero:
18


In [14]:
source_name_mapping = {
    "source_score_vehicular_traffic": "Vehicular traffic",
    "source_score_industrial_combustion": "Industrial combustion",
    "source_score_construction__road_dust": "Construction / road dust",
    "source_score_biomass__fine_particle_combustion": "Biomass / fine-particle combustion",
    "source_score_photochemical_smog": "Photochemical smog"
}

source_score_columns = [
    col for col in station_df.columns
    if col.startswith("source_score_")
]

station_df[source_score_columns] = station_df[source_score_columns].apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0)

station_df["primary_source_score_column"] = station_df[source_score_columns].idxmax(axis=1)

station_df["primary_pollution_source"] = station_df[
    "primary_source_score_column"
].map(source_name_mapping)

station_df["primary_source_score"] = station_df[source_score_columns].max(axis=1)

station_df["secondary_source_score"] = station_df[source_score_columns].apply(
    lambda row: row.sort_values(ascending=False).iloc[1],
    axis=1
)

station_df["source_score_gap"] = (
    station_df["primary_source_score"]
    - station_df["secondary_source_score"]
).round(2)

station_df.loc[
    station_df[source_score_columns].sum(axis=1) == 0,
    "primary_pollution_source"
] = "Insufficient pollutant signal"

print("Primary pollution source assigned.")

display(
    station_df[
        [
            "state",
            "city",
            "station_name",
            "reported_aqi",
            "reported_aqi_category",
            "primary_pollution_source",
            "primary_source_score",
            "secondary_source_score",
            "source_score_gap"
        ]
    ].head(20)
)

Primary pollution source assigned.


,state,city,station_name,reported_aqi,reported_aqi_category,primary_pollution_source,primary_source_score,secondary_source_score,source_score_gap
0,Andhra Pradesh,Amaravati,"Secretariat, Amaravati - APPCB",500.00,Severe,Construction / road dust,38.40,34.56,3.84
1,Andhra Pradesh,Anantapur,"Gulzarpet, Anantapur - APPCB",329.70,Very Poor,Biomass / fine-particle combustion,35.37,29.84,5.53
2,Andhra Pradesh,Chittoor,"Gangineni Cheruvu, Chittoor - APPCB",323.85,Very Poor,Biomass / fine-particle combustion,36.34,33.72,2.62
3,Andhra Pradesh,Eluru,"District Court, Eluru - APPCB",306.27,Very Poor,Biomass / fine-particle combustion,34.09,26.13,7.96
4,Andhra Pradesh,Guntur,"Rajendra Nagar North, Guntur - APPCB",271.30,Poor,Photochemical smog,32.25,31.11,1.14
5,Andhra Pradesh,Kadapa,"Yerramukkapalli, Kadapa - APPCB",364.85,Very Poor,Photochemical smog,33.26,31.25,2.01
6,Andhra Pradesh,Kurnool,"Sita Rama Nagar, Kurnool - APPCB",174.94,Moderate,Photochemical smog,34.25,26.62,7.63
7,Andhra Pradesh,Machilipatnam,"Srinivas Nagar Colony, Machilipatnam - APPCB",317.99,Very Poor,Photochemical smog,30.77,29.10,1.67
8,Andhra Pradesh,Nellore,"Ambedkar Nagar, Nellore - APPCB",412.83,Severe,Photochemical smog,36.00,31.63,4.37
9,Andhra Pradesh,Rajamahendravaram,"Anand Kala Kshetram, Rajamahendravaram - APPCB",353.14,Very Poor,Biomass / fine-particle combustion,30.66,27.65,3.01


In [15]:
def confidence_category(score):
    if pd.isna(score):
        return "Unknown"
    elif score < 40:
        return "Low confidence"
    elif score < 65:
        return "Moderate confidence"
    elif score < 80:
        return "High confidence"
    else:
        return "Very high confidence"


station_df["source_attribution_confidence"] = (
    0.70 * station_df["primary_source_score"]
    + 0.30 * station_df["source_score_gap"]
).round(2)

station_df["source_attribution_confidence"] = station_df[
    "source_attribution_confidence"
].clip(0, 100)

station_df["source_attribution_confidence_category"] = station_df[
    "source_attribution_confidence"
].apply(confidence_category)

print("Confidence calculated.")

print(station_df["source_attribution_confidence_category"].value_counts())

display(
    station_df[
        [
            "primary_pollution_source",
            "primary_source_score",
            "source_score_gap",
            "source_attribution_confidence",
            "source_attribution_confidence_category"
        ]
    ].head()
)

Confidence calculated.
source_attribution_confidence_category
Low confidence         447
Moderate confidence     53
Name: count, dtype: int64


,primary_pollution_source,primary_source_score,source_score_gap,source_attribution_confidence,source_attribution_confidence_category
0,Construction / road dust,38.40,3.84,28.03,Low confidence
1,Biomass / fine-particle combustion,35.37,5.53,26.42,Low confidence
2,Biomass / fine-particle combustion,36.34,2.62,26.22,Low confidence
3,Biomass / fine-particle combustion,34.09,7.96,26.25,Low confidence
4,Photochemical smog,32.25,1.14,22.92,Low confidence


In [16]:
def generate_source_evidence(row):
    source = row.get("primary_pollution_source", "Unknown")

    evidence = []

    if source == "Vehicular traffic":
        evidence.append("elevated NO2/CO signal")
        evidence.append("road-density or urban-pressure influence")
        if row.get("is_traffic_landuse", 0) == 1:
            evidence.append("traffic/commercial land-use proxy")

    elif source == "Industrial combustion":
        evidence.append("SO2 and combustion-related pollutant influence")
        if row.get("is_industrial_landuse", 0) == 1:
            evidence.append("industrial land-use proxy")

    elif source == "Construction / road dust":
        evidence.append("PM10-heavy particulate pattern")
        evidence.append("road dust / coarse-particle likelihood")
        if row.get("road_density_proxy", 0) >= 60:
            evidence.append("high road-density proxy")

    elif source == "Biomass / fine-particle combustion":
        evidence.append("PM2.5-heavy fine-particle pattern")
        evidence.append("CO and stagnant-weather influence")
        if row.get("weather_trapping_score", 0) >= 60:
            evidence.append("high weather trapping")

    elif source == "Photochemical smog":
        evidence.append("O3 and temperature-related signal")
        evidence.append("urban photochemical formation likelihood")

    else:
        evidence.append("insufficient source signal")

    return "; ".join(evidence)


station_df["source_evidence"] = station_df.apply(
    generate_source_evidence,
    axis=1
)

display(
    station_df[
        [
            "primary_pollution_source",
            "source_evidence"
        ]
    ].head(20)
)

,primary_pollution_source,source_evidence
0,Construction / road dust,PM10-heavy particulate pattern; road dust / co...
1,Biomass / fine-particle combustion,PM2.5-heavy fine-particle pattern; CO and stag...
2,Biomass / fine-particle combustion,PM2.5-heavy fine-particle pattern; CO and stag...
3,Biomass / fine-particle combustion,PM2.5-heavy fine-particle pattern; CO and stag...
4,Photochemical smog,O3 and temperature-related signal; urban photo...
5,Photochemical smog,O3 and temperature-related signal; urban photo...
6,Photochemical smog,O3 and temperature-related signal; urban photo...
7,Photochemical smog,O3 and temperature-related signal; urban photo...
8,Photochemical smog,O3 and temperature-related signal; urban photo...
9,Biomass / fine-particle combustion,PM2.5-heavy fine-particle pattern; CO and stag...


In [17]:
def check_landuse_alignment(row):
    source = row.get("primary_pollution_source", "")
    landuse = str(row.get("landuse_type_proxy", "")).lower()

    if source == "Vehicular traffic":
        if "traffic" in landuse or "commercial" in landuse or "dense urban" in landuse:
            return "Aligned"
        return "Partially aligned"

    if source == "Industrial combustion":
        if "industrial" in landuse:
            return "Aligned"
        return "Not strongly aligned"

    if source == "Construction / road dust":
        if "traffic" in landuse or "urban" in landuse or "road" in landuse:
            return "Partially aligned"
        return "Not strongly aligned"

    if source == "Biomass / fine-particle combustion":
        if "general urban" in landuse or "semi-urban" in landuse or "dense urban" in landuse:
            return "Partially aligned"
        return "Not strongly aligned"

    if source == "Photochemical smog":
        if "urban" in landuse or "traffic" in landuse or "commercial" in landuse:
            return "Partially aligned"
        return "Not strongly aligned"

    return "Unknown"


station_df["landuse_source_alignment"] = station_df.apply(
    check_landuse_alignment,
    axis=1
)

print("Land-use alignment distribution:")
print(station_df["landuse_source_alignment"].value_counts())

display(
    station_df[
        [
            "landuse_type_proxy",
            "primary_pollution_source",
            "landuse_source_alignment"
        ]
    ].head(20)
)

Land-use alignment distribution:
landuse_source_alignment
Partially aligned       329
Aligned                 147
Unknown                  18
Not strongly aligned      6
Name: count, dtype: int64


,landuse_type_proxy,primary_pollution_source,landuse_source_alignment
0,General urban / semi-urban proxy,Construction / road dust,Partially aligned
1,General urban / semi-urban proxy,Biomass / fine-particle combustion,Partially aligned
2,General urban / semi-urban proxy,Biomass / fine-particle combustion,Partially aligned
3,General urban / semi-urban proxy,Biomass / fine-particle combustion,Partially aligned
4,General urban / semi-urban proxy,Photochemical smog,Partially aligned
5,General urban / semi-urban proxy,Photochemical smog,Partially aligned
6,General urban / semi-urban proxy,Photochemical smog,Partially aligned
7,General urban / semi-urban proxy,Photochemical smog,Partially aligned
8,General urban / semi-urban proxy,Photochemical smog,Partially aligned
9,General urban / semi-urban proxy,Biomass / fine-particle combustion,Partially aligned


In [18]:
def generate_recommendation(row):
    source = row.get("primary_pollution_source", "")
    category = row.get("reported_aqi_category", "Unknown")

    urgency = "Monitor"
    if category in ["Poor", "Very Poor", "Severe"]:
        urgency = "High priority"
    elif category == "Moderate":
        urgency = "Medium priority"

    if source == "Vehicular traffic":
        action = (
            "Prioritize traffic-flow management, public transport promotion, "
            "anti-idling enforcement, and congestion reduction near the station."
        )

    elif source == "Industrial combustion":
        action = (
            "Prioritize industrial emission inspection, stack monitoring, "
            "fuel-quality checks, and enforcement near industrial clusters."
        )

    elif source == "Construction / road dust":
        action = (
            "Prioritize road dust suppression, construction-site covering, "
            "mechanical sweeping, and enforcement of dust-control rules."
        )

    elif source == "Biomass / fine-particle combustion":
        action = (
            "Prioritize open-burning control, biomass-combustion monitoring, "
            "fine-particle exposure advisories, and local source inspection."
        )

    elif source == "Photochemical smog":
        action = (
            "Prioritize NOx/VOC precursor control, traffic-emission reduction, "
            "and ozone exposure advisories during sunny high-temperature periods."
        )

    else:
        action = "Collect more pollutant and land-use data before assigning intervention."

    return f"{urgency}: {action}"


station_df["recommended_intervention"] = station_df.apply(
    generate_recommendation,
    axis=1
)

display(
    station_df[
        [
            "reported_aqi_category",
            "primary_pollution_source",
            "recommended_intervention"
        ]
    ].head(20)
)

,reported_aqi_category,primary_pollution_source,recommended_intervention
0,Severe,Construction / road dust,High priority: Prioritize road dust suppressio...
1,Very Poor,Biomass / fine-particle combustion,High priority: Prioritize open-burning control...
2,Very Poor,Biomass / fine-particle combustion,High priority: Prioritize open-burning control...
3,Very Poor,Biomass / fine-particle combustion,High priority: Prioritize open-burning control...
4,Poor,Photochemical smog,High priority: Prioritize NOx/VOC precursor co...
5,Very Poor,Photochemical smog,High priority: Prioritize NOx/VOC precursor co...
6,Moderate,Photochemical smog,Medium priority: Prioritize NOx/VOC precursor ...
7,Very Poor,Photochemical smog,High priority: Prioritize NOx/VOC precursor co...
8,Severe,Photochemical smog,High priority: Prioritize NOx/VOC precursor co...
9,Very Poor,Biomass / fine-particle combustion,High priority: Prioritize open-burning control...


In [19]:
print("Primary source distribution:")
print(station_df["primary_pollution_source"].value_counts(dropna=False))

print("\nSource confidence distribution:")
print(station_df["source_attribution_confidence_category"].value_counts(dropna=False))

print("\nSource by AQI category:")
display(
    pd.crosstab(
        station_df["reported_aqi_category"],
        station_df["primary_pollution_source"]
    )
)

Primary source distribution:
primary_pollution_source
Biomass / fine-particle combustion    160
Vehicular traffic                     134
Photochemical smog                    125
Construction / road dust               46
Insufficient pollutant signal          18
Industrial combustion                  17
Name: count, dtype: int64

Source confidence distribution:
source_attribution_confidence_category
Low confidence         447
Moderate confidence     53
Name: count, dtype: int64

Source by AQI category:


primary_pollution_source,Biomass / fine-particle combustion,Construction / road dust,Industrial combustion,Insufficient pollutant signal,Photochemical smog,Vehicular traffic
reported_aqi_category,,,,,,
Good,0,1,0,2,0,0
Moderate,16,7,0,0,18,8
Poor,25,6,3,0,36,14
Satisfactory,0,7,0,1,2,2
Severe,53,11,7,1,20,55
Unknown,0,1,1,13,7,0
Very Poor,66,13,6,1,42,55


In [20]:
display(
    station_df.sort_values(
        ["reported_aqi", "source_attribution_confidence"],
        ascending=False
    )[
        [
            "state",
            "city",
            "station_name",
            "reported_aqi",
            "reported_aqi_category",
            "primary_pollution_source",
            "source_attribution_confidence",
            "source_attribution_confidence_category",
            "source_evidence",
            "landuse_type_proxy",
            "landuse_source_alignment",
            "recommended_intervention"
        ]
    ].head(25)
)

,state,city,station_name,reported_aqi,reported_aqi_category,primary_pollution_source,source_attribution_confidence,source_attribution_confidence_category,source_evidence,landuse_type_proxy,landuse_source_alignment,recommended_intervention
440,Uttar Pradesh,Khora,"Prashant Garden, Khora - UPPCB",500.0,Severe,Biomass / fine-particle combustion,62.73,Moderate confidence,PM2.5-heavy fine-particle pattern; CO and stag...,General urban / semi-urban proxy,Partially aligned,High priority: Prioritize open-burning control...
180,Karnataka,Bengaluru,"City Railway Station, Bengaluru - KSPCB",500.0,Severe,Vehicular traffic,59.48,Moderate confidence,elevated NO2/CO signal; road-density or urban-...,Traffic / commercial proxy,Aligned,High priority: Prioritize traffic-flow managem...
106,Delhi,Delhi,"North Campus, DU, Delhi - IITM",500.0,Severe,Vehicular traffic,57.06,Moderate confidence,elevated NO2/CO signal; road-density or urban-...,Dense urban proxy,Aligned,High priority: Prioritize traffic-flow managem...
82,Delhi,Delhi,"CRRI Mathura Road, Delhi - IITM",500.0,Severe,Vehicular traffic,56.60,Moderate confidence,elevated NO2/CO signal; road-density or urban-...,Traffic / commercial proxy,Aligned,High priority: Prioritize traffic-flow managem...
89,Delhi,Delhi,"IGI Airport (T3), Delhi - IITM",500.0,Severe,Vehicular traffic,56.52,Moderate confidence,elevated NO2/CO signal; road-density or urban-...,Dense urban proxy,Aligned,High priority: Prioritize traffic-flow managem...
84,Delhi,Delhi,"Chandni Chowk, Delhi - IITM",500.0,Severe,Vehicular traffic,51.17,Moderate confidence,elevated NO2/CO signal; road-density or urban-...,Traffic / commercial proxy,Aligned,High priority: Prioritize traffic-flow managem...
342,Punjab,Rupnagar,"Ratanpura, Rupnagar - Ambuja Cements",500.0,Severe,Biomass / fine-particle combustion,49.84,Moderate confidence,PM2.5-heavy fine-particle pattern; CO and stag...,General urban / semi-urban proxy,Partially aligned,High priority: Prioritize open-burning control...
33,Bihar,Bihar Sharif,"D M Colony, Bihar Sharif - BSPCB",500.0,Severe,Biomass / fine-particle combustion,49.10,Moderate confidence,PM2.5-heavy fine-particle pattern; CO and stag...,General urban / semi-urban proxy,Partially aligned,High priority: Prioritize open-burning control...
126,Gujarat,Ahmedabad,"SAC ISRO Satellite, Ahmedabad - IITM",500.0,Severe,Vehicular traffic,46.34,Moderate confidence,elevated NO2/CO signal; road-density or urban-...,Dense urban proxy,Aligned,High priority: Prioritize traffic-flow managem...
299,Maharashtra,Pune,"Dhankawadi, Pune - IITM",500.0,Severe,Vehicular traffic,45.36,Moderate confidence,elevated NO2/CO signal; road-density or urban-...,Dense urban proxy,Aligned,High priority: Prioritize traffic-flow managem...


In [21]:
city_source_summary = (
    station_df
    .groupby(["state", "city"], as_index=False)
    .agg(
        station_count=("station_id", "nunique"),
        mean_reported_aqi=("reported_aqi", "mean"),
        max_reported_aqi=("reported_aqi", "max"),
        mean_source_confidence=("source_attribution_confidence", "mean"),
        mean_environmental_risk_score=("environmental_risk_score", "mean")
    )
)

city_primary_source = (
    station_df
    .groupby(["state", "city"])["primary_pollution_source"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={"primary_pollution_source": "dominant_city_source"})
)

city_source_summary = city_source_summary.merge(
    city_primary_source,
    on=["state", "city"],
    how="left"
)

for col in [
    "mean_reported_aqi",
    "max_reported_aqi",
    "mean_source_confidence",
    "mean_environmental_risk_score"
]:
    city_source_summary[col] = city_source_summary[col].round(2)

CITY_SOURCE_SUMMARY_PATH = PROCESSED_DIR / "city_source_attribution_summary.csv"

city_source_summary.to_csv(
    CITY_SOURCE_SUMMARY_PATH,
    index=False
)

print("City source summary saved:")
print(CITY_SOURCE_SUMMARY_PATH)

display(
    city_source_summary.sort_values(
        "mean_reported_aqi",
        ascending=False
    ).head(20)
)

City source summary saved:
C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\city_source_attribution_summary.csv


,state,city,station_count,mean_reported_aqi,max_reported_aqi,mean_source_confidence,mean_environmental_risk_score,dominant_city_source
0,Andhra Pradesh,Amaravati,1,500.0,500.0,28.03,51.86,Construction / road dust
14,Assam,Byrnihat,1,500.0,500.0,41.75,63.47,Biomass / fine-particle combustion
26,Bihar,Bihar Sharif,1,500.0,500.0,49.10,62.99,Biomass / fine-particle combustion
28,Bihar,Chhapra,1,500.0,500.0,40.91,58.54,Biomass / fine-particle combustion
31,Bihar,Katihar,1,500.0,500.0,37.43,59.46,Biomass / fine-particle combustion
38,Bihar,Purnia,1,500.0,500.0,35.63,56.22,Biomass / fine-particle combustion
64,Haryana,Ambala,1,500.0,500.0,35.99,57.20,Biomass / fine-particle combustion
56,Gujarat,Bhavnagar,1,500.0,500.0,37.08,62.75,Biomass / fine-particle combustion
58,Gujarat,Mehsana,1,500.0,500.0,33.67,54.56,Biomass / fine-particle combustion
59,Gujarat,Rajkot,1,500.0,500.0,34.19,54.12,Biomass / fine-particle combustion


In [22]:
final_columns = [
    "station_id",
    "station_name",
    "city",
    "state",
    "lat",
    "lon",
    "timestamp",

    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "CO",
    "O3",
    "NH3",

    "reported_aqi",
    "reported_aqi_category",
    "reported_dominant_pollutant",
    "aqi_is_valid",
    "aqi_quality_flag",

    "temperature_2m",
    "relative_humidity_2m",
    "rain",
    "wind_speed_10m",
    "cloud_cover",
    "weather_trapping_score",
    "weather_trapping_category",

    "landuse_type_proxy",
    "landuse_data_source",
    "urban_pressure_score",
    "road_density_proxy",
    "environmental_risk_score",

    "pm10_pm25_ratio",
    "pm25_pm10_ratio",
    "no2_co_ratio",
    "o3_no2_ratio",

    "source_score_vehicular_traffic",
    "source_score_industrial_combustion",
    "source_score_construction__road_dust",
    "source_score_biomass__fine_particle_combustion",
    "source_score_photochemical_smog",

    "primary_pollution_source",
    "primary_source_score",
    "secondary_source_score",
    "source_score_gap",
    "source_attribution_confidence",
    "source_attribution_confidence_category",
    "source_evidence",
    "landuse_source_alignment",
    "recommended_intervention"
]

existing_final_columns = [
    col for col in final_columns
    if col in station_df.columns
]

station_source_final = station_df[existing_final_columns].copy()

station_source_final["notebook_03_processed_time"] = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

print("Final station source-attribution shape:", station_source_final.shape)
display(station_source_final.head())

Final station source-attribution shape: (500, 50)


,station_id,station_name,city,state,lat,lon,timestamp,PM2.5,PM10,NO2,...,primary_pollution_source,primary_source_score,secondary_source_score,source_score_gap,source_attribution_confidence,source_attribution_confidence_category,source_evidence,landuse_source_alignment,recommended_intervention,notebook_03_processed_time
0,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-16 05:00:00,54.0,172.0,32.0,...,Construction / road dust,38.40,34.56,3.84,28.03,Low confidence,PM10-heavy particulate pattern; road dust / co...,Partially aligned,High priority: Prioritize road dust suppressio...,2026-07-16 10:55:13
1,andhra_pradesh__anantapur__gulzarpet_anantapur...,"Gulzarpet, Anantapur - APPCB",Anantapur,Andhra Pradesh,14.675886,77.593027,2026-07-16 05:00:00,85.0,76.0,16.0,...,Biomass / fine-particle combustion,35.37,29.84,5.53,26.42,Low confidence,PM2.5-heavy fine-particle pattern; CO and stag...,Partially aligned,High priority: Prioritize open-burning control...,2026-07-16 10:55:13
2,andhra_pradesh__chittoor__gangineni_cheruvu_ch...,"Gangineni Cheruvu, Chittoor - APPCB",Chittoor,Andhra Pradesh,13.204880,79.097889,2026-07-16 05:00:00,70.0,64.0,31.0,...,Biomass / fine-particle combustion,36.34,33.72,2.62,26.22,Low confidence,PM2.5-heavy fine-particle pattern; CO and stag...,Partially aligned,High priority: Prioritize open-burning control...,2026-07-16 10:55:13
3,andhra_pradesh__eluru__district_court_eluru_appcb,"District Court, Eluru - APPCB",Eluru,Andhra Pradesh,16.711754,81.092095,2026-07-16 05:00:00,49.0,49.0,22.0,...,Biomass / fine-particle combustion,34.09,26.13,7.96,26.25,Low confidence,PM2.5-heavy fine-particle pattern; CO and stag...,Partially aligned,High priority: Prioritize open-burning control...,2026-07-16 10:55:13
4,andhra_pradesh__guntur__rajendra_nagar_north_g...,"Rajendra Nagar North, Guntur - APPCB",Guntur,Andhra Pradesh,16.316553,80.413302,2026-07-16 05:00:00,52.0,52.0,42.0,...,Photochemical smog,32.25,31.11,1.14,22.92,Low confidence,O3 and temperature-related signal; urban photo...,Partially aligned,High priority: Prioritize NOx/VOC precursor co...,2026-07-16 10:55:13


In [23]:
SOURCE_OUTPUT_PATH = PROCESSED_DIR / "station_source_attributed.csv"

station_source_final.to_csv(
    SOURCE_OUTPUT_PATH,
    index=False
)

print("Saved station source-attributed output:")
print(SOURCE_OUTPUT_PATH)
print("Rows:", len(station_source_final))
print("Columns:", len(station_source_final.columns))

Saved station source-attributed output:
C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\station_source_attributed.csv
Rows: 500
Columns: 50


In [24]:
run_summary = {
    "notebook": "03_source_attribution_landuse.ipynb",
    "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "input_file": str(INPUT_PATH),
    "input_rows": int(len(station_df)),
    "output_rows": int(len(station_source_final)),
    "unique_stations": int(station_source_final["station_id"].nunique()),
    "unique_cities": int(station_source_final["city"].nunique()),
    "unique_states": int(station_source_final["state"].nunique()),
    "source_distribution": station_source_final["primary_pollution_source"].value_counts().to_dict(),
    "confidence_distribution": station_source_final["source_attribution_confidence_category"].value_counts().to_dict(),
    "output_file": str(SOURCE_OUTPUT_PATH),
    "city_summary_file": str(CITY_SOURCE_SUMMARY_PATH)
}

RUN_SUMMARY_PATH = REPORTS_DIR / "notebook_03_run_summary.json"

with open(RUN_SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=4)

print("Run summary saved:")
print(RUN_SUMMARY_PATH)

print(json.dumps(run_summary, indent=4))

Run summary saved:
C:\Users\Lenovo\Desktop\CleanAir_AI\reports\notebook_03_run_summary.json
{
    "notebook": "03_source_attribution_landuse.ipynb",
    "run_time": "2026-07-16 10:55:14",
    "input_file": "C:\\Users\\Lenovo\\Desktop\\CleanAir_AI\\data\\processed\\station_weather_geospatial.csv",
    "input_rows": 500,
    "output_rows": 500,
    "unique_stations": 500,
    "unique_cities": 264,
    "unique_states": 29,
    "source_distribution": {
        "Biomass / fine-particle combustion": 160,
        "Vehicular traffic": 134,
        "Photochemical smog": 125,
        "Construction / road dust": 46,
        "Insufficient pollutant signal": 18,
        "Industrial combustion": 17
    },
    "confidence_distribution": {
        "Low confidence": 447,
        "Moderate confidence": 53
    },
    "output_file": "C:\\Users\\Lenovo\\Desktop\\CleanAir_AI\\data\\processed\\station_source_attributed.csv",
    "city_summary_file": "C:\\Users\\Lenovo\\Desktop\\CleanAir_AI\\data\\processed\\

In [25]:
required_output_files = [
    SOURCE_OUTPUT_PATH,
    CITY_SOURCE_SUMMARY_PATH,
    RUN_SUMMARY_PATH
]

missing_files = [
    path for path in required_output_files
    if not path.exists()
]

if missing_files:
    print("Missing files:")

    for path in missing_files:
        print(path)

    raise FileNotFoundError("Some expected output files were not created.")

print("Notebook 03 completed successfully.")
print("Created files:")

for path in required_output_files:
    print("-", path)
    

Notebook 03 completed successfully.
Created files:
- C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\station_source_attributed.csv
- C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\city_source_attribution_summary.csv
- C:\Users\Lenovo\Desktop\CleanAir_AI\reports\notebook_03_run_summary.json
